# 03 - Mushroom classification met Histogram Gradient Boosting

**Auteur:** Brent Van Gelder  
**Dataset:** `mushroom_project_dataset.csv`  
**Doel:** voorspellen of een paddenstoel eetbaar (`e`) of giftig/onbekend (`p`) is.

> Dit is een onderwijsmodel en mag nooit worden gebruikt om echte paddenstoelen veilig te verklaren.

Deze notebook gebruikt dezelfde datacontroles, uitgesloten ruiskolommen, random seed, 70/15/15-split en evaluatiemetrics als de Random Forest-notebook. Het model krijgt dezelfde geïmputeerde en one-hot gecodeerde data, maar heeft geen schaling nodig.

## Technologie

Histogram Gradient Boosting bouwt bomen na elkaar. Elke nieuwe boom probeert fouten van de bestaande verzameling te corrigeren. Het kan niet-lineaire verbanden en interacties leren en is duidelijk anders dan zowel Logistic Regression als Random Forest.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_predict,
    train_test_split,
)
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import make_scorer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

## 1. Data en vaste instellingen

De meegeleverde project-CSV wordt lokaal geladen. De twee expliciet als `jumbled_noise` benoemde kolommen worden in alle modelnotebooks uitgesloten. We controleren labels, ontbrekende waarden en duplicaten voordat het model wordt gebouwd.

In [ ]:
RANDOM_STATE = 42
DATA_PATH = Path("mushroom_project_dataset.csv")
TARGET = "class"
EDIBLE_LABEL = "e"
POISONOUS_LABEL = "p"
RUN_FINAL_TEST = False

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset niet gevonden: {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)
if TARGET not in df.columns:
    raise ValueError(f"Doelkolom '{TARGET}' ontbreekt.")
if df[TARGET].isna().any():
    raise ValueError("De doelkolom bevat ontbrekende waarden.")
if set(df[TARGET].unique()) != {EDIBLE_LABEL, POISONOUS_LABEL}:
    raise ValueError(f"Onverwachte labels: {set(df[TARGET].unique())}")

noise_columns = [column for column in df.columns if column.startswith("jumbled_noise_")]
model_df = df.drop(columns=noise_columns).copy()

summary = pd.DataFrame({
    "datatype": model_df.dtypes.astype(str),
    "ontbrekend": model_df.isna().sum(),
    "ontbrekend_percentage": (100 * model_df.isna().mean()).round(1),
    "unieke_waarden": model_df.nunique(dropna=True),
}).sort_values("ontbrekend_percentage", ascending=False)

print(f"Rijen: {len(model_df):,}")
print(f"Exact dubbele rijen: {model_df.duplicated().sum()}")
print(f"Uitgesloten ruiskolommen: {noise_columns}")
display(model_df[TARGET].value_counts().rename_axis("klasse").to_frame("aantal"))
display(summary)

## 2. Dezelfde train-, validatie- en testset

Met `random_state=42` en dezelfde tweestapssplit krijgt iedere notebook exact dezelfde 70% trainingsrijen, 15% validatierijen en 15% testrijen. `stratify` bewaart de klasseverhouding. De testset blijft vergrendeld totdat alle keuzes zijn gemaakt.

In [ ]:
X = model_df.drop(columns=TARGET)
y = model_df[TARGET]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

split_summary = pd.DataFrame({
    "rijen": [len(X_train), len(X_validation), len(X_test)],
    "aandeel_dataset": [len(X_train) / len(X), len(X_validation) / len(X), len(X_test) / len(X)],
    "aandeel_giftig": [
        (y_train == POISONOUS_LABEL).mean(),
        (y_validation == POISONOUS_LABEL).mean(),
        (y_test == POISONOUS_LABEL).mean(),
    ],
}, index=["train", "validatie", "test"])
display(split_summary.style.format({
    "aandeel_dataset": "{:.1%}",
    "aandeel_giftig": "{:.1%}",
}))

## 3. Cleaning, preprocessing en model

We gebruiken dezelfde numerieke mediaan, missingness-indicatoren, expliciete categorie voor ontbrekende categorische waarden en veilige one-hot encoding. Schaling is niet nodig voor beslissingsbomen. Interne early stopping voorkomt dat onnodig veel boosting-rondes worden uitgevoerd.

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median", add_indicator=True), numeric_features),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="__missing__")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_features),
], verbose_feature_names_out=False)

baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", HistGradientBoostingClassifier(
        learning_rate=0.08,
        max_iter=300,
        max_leaf_nodes=31,
        min_samples_leaf=20,
        l2_regularization=0.1,
        class_weight="balanced",
        early_stopping=True,
        validation_fraction=0.15,
        random_state=RANDOM_STATE,
    )),
])

baseline_model.fit(X_train, y_train)
print("Histogram Gradient Boosting getraind op de trainingsset.")

## 4. Dezelfde evaluatiemetrics

De giftige klasse `p` is de positieve klasse. F2 weegt recall zwaarder dan precision, omdat een giftige paddenstoel ten onrechte als eetbaar classificeren de ernstigste fout is. We rapporteren daarnaast accuracy, balanced accuracy, F1, ROC-AUC en average precision.

In [ ]:
def evaluate_classifier(fitted_model, X_eval, y_eval, split_name, threshold=0.50):
    poisonous_index = list(fitted_model.classes_).index(POISONOUS_LABEL)
    poisonous_probability = fitted_model.predict_proba(X_eval)[:, poisonous_index]
    predictions = np.where(
        poisonous_probability >= threshold, POISONOUS_LABEL, EDIBLE_LABEL
    )
    y_binary = (y_eval == POISONOUS_LABEL).astype(int)

    metrics = pd.Series({
        "accuracy": accuracy_score(y_eval, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_eval, predictions),
        "precision_giftig": precision_score(
            y_eval, predictions, pos_label=POISONOUS_LABEL, zero_division=0
        ),
        "recall_giftig": recall_score(
            y_eval, predictions, pos_label=POISONOUS_LABEL, zero_division=0
        ),
        "f1_giftig": f1_score(
            y_eval, predictions, pos_label=POISONOUS_LABEL, zero_division=0
        ),
        "f2_giftig": fbeta_score(
            y_eval, predictions, beta=2, pos_label=POISONOUS_LABEL, zero_division=0
        ),
        "roc_auc": roc_auc_score(y_binary, poisonous_probability),
        "average_precision": average_precision_score(y_binary, poisonous_probability),
    }, name=split_name)

    print(f"Drempel voor giftig: {threshold:.3f}")
    display(metrics.to_frame().T.style.format("{:.3f}"))
    print(classification_report(
        y_eval,
        predictions,
        labels=[EDIBLE_LABEL, POISONOUS_LABEL],
        target_names=["eetbaar", "giftig/onbekend"],
        zero_division=0,
    ))
    ConfusionMatrixDisplay.from_predictions(
        y_eval,
        predictions,
        labels=[EDIBLE_LABEL, POISONOUS_LABEL],
        display_labels=["eetbaar", "giftig/onbekend"],
        cmap="Blues",
    )
    plt.title(f"Confusion matrix - {split_name}")
    plt.grid(False)
    plt.show()
    return metrics

## 5. Validatie

We evalueren uitsluitend op de aparte validatieset. Het model gebruikt daarnaast tijdens training een intern deel van de trainingsset voor early stopping; dat is niet de externe validatie- of testset.

In [ ]:
validation_metrics = evaluate_classifier(
    baseline_model, X_validation, y_validation, "gradient_boosting_baseline_validatie"
)

## 6. Hyperparameters tunen met RandomizedSearchCV

We testen 24 reproduceerbaar gelote configuraties met vijfvoudige gestratificeerde cross-validatie binnen uitsluitend de trainingsset. F2 voor de giftige klasse is de hoofdscore. De aparte validatieset wordt pas na de zoektocht gebruikt om te controleren of de winst generaliseert.

In [ ]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
f2_scorer = make_scorer(
    fbeta_score, beta=2, pos_label=POISONOUS_LABEL, zero_division=0
)
recall_poisonous_scorer = make_scorer(
    recall_score, pos_label=POISONOUS_LABEL, zero_division=0
)

parameter_distributions = {
    "classifier__learning_rate": [0.03, 0.05, 0.08, 0.12, 0.18],
    "classifier__max_iter": [150, 250, 400, 600],
    "classifier__max_leaf_nodes": [7, 15, 31, 63],
    "classifier__max_depth": [None, 4, 6, 10],
    "classifier__min_samples_leaf": [10, 20, 30, 50],
    "classifier__l2_regularization": [0.0, 0.1, 1.0, 5.0, 10.0],
    "classifier__class_weight": [None, "balanced"],
}

random_search = RandomizedSearchCV(
    estimator=clone(baseline_model),
    param_distributions=parameter_distributions,
    n_iter=24,
    scoring={
        "f2_giftig": f2_scorer,
        "recall_giftig": recall_poisonous_scorer,
        "balanced_accuracy": "balanced_accuracy",
    },
    refit="f2_giftig",
    cv=cv_strategy,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    return_train_score=True,
    verbose=1,
    error_score="raise",
)

random_search.fit(X_train, y_train)
tuned_model = random_search.best_estimator_
print(f"Beste gemiddelde CV F2-score: {random_search.best_score_:.3f}")
display(pd.Series(random_search.best_params_, name="waarde").to_frame())

### Baseline en getuned model vergelijken op validatie

We tonen de tien beste cross-validatieconfiguraties en evalueren het beste getunede model daarna één keer op de aparte validatieset. De hoogste validatie-F2 bepaalt welk Gradient Boosting-model kandidaat blijft.

In [ ]:
cv_results = (
    pd.DataFrame(random_search.cv_results_)
    .sort_values("rank_test_f2_giftig")
)
display(cv_results[[
    "rank_test_f2_giftig",
    "mean_test_f2_giftig",
    "std_test_f2_giftig",
    "mean_test_recall_giftig",
    "mean_test_balanced_accuracy",
    "mean_train_f2_giftig",
    "params",
]].head(10))

tuned_validation_metrics = evaluate_classifier(
    tuned_model, X_validation, y_validation, "gradient_boosting_getuned_validatie"
)
validation_comparison = pd.DataFrame([validation_metrics, tuned_validation_metrics])
display(validation_comparison.style.format("{:.3f}"))

if tuned_validation_metrics["f2_giftig"] >= validation_metrics["f2_giftig"]:
    selected_model = tuned_model
    selected_model_name = "getuned Histogram Gradient Boosting"
else:
    selected_model = baseline_model
    selected_model_name = "baseline Histogram Gradient Boosting"

print(f"Geselecteerde kandidaat op basis van validatie-F2: {selected_model_name}")

## 7. Beslisdrempel kiezen met out-of-fold voorspellingen

De standaarddrempel is 0,50. We maken eerst out-of-fold kansen voor elke trainingsrij: iedere rij wordt voorspeld door een model dat die rij niet tijdens het fitten zag. Daarna kiezen we de drempel met de hoogste accuracy onder de harde voorwaarde dat recall voor giftig minimaal 85% is. De aparte validatieset blijft buiten de drempelkeuze.

In [ ]:
TARGET_POISONOUS_RECALL = 0.85

oof_probability_matrix = cross_val_predict(
    clone(selected_model),
    X_train,
    y_train,
    cv=cv_strategy,
    method="predict_proba",
    n_jobs=1,
)
poisonous_index = list(selected_model.classes_).index(POISONOUS_LABEL)
oof_poisonous_probability = oof_probability_matrix[:, poisonous_index]

threshold_rows = []
for threshold in np.round(np.arange(0.05, 0.951, 0.01), 2):
    predictions = np.where(
        oof_poisonous_probability >= threshold, POISONOUS_LABEL, EDIBLE_LABEL
    )
    threshold_rows.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_train, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_train, predictions),
        "precision_giftig": precision_score(
            y_train, predictions, pos_label=POISONOUS_LABEL, zero_division=0
        ),
        "recall_giftig": recall_score(
            y_train, predictions, pos_label=POISONOUS_LABEL, zero_division=0
        ),
        "f2_giftig": fbeta_score(
            y_train, predictions, beta=2, pos_label=POISONOUS_LABEL, zero_division=0
        ),
    })

threshold_results = pd.DataFrame(threshold_rows)
eligible_thresholds = threshold_results.query(
    "recall_giftig >= @TARGET_POISONOUS_RECALL"
)
if eligible_thresholds.empty:
    raise RuntimeError("Geen drempel voldoet aan de minimale recall.")

selected_threshold_row = eligible_thresholds.sort_values(
    ["accuracy", "precision_giftig", "f2_giftig"],
    ascending=False,
).iloc[0]
safety_threshold = float(selected_threshold_row["threshold"])
accuracy_threshold_row = threshold_results.sort_values(
    ["accuracy", "balanced_accuracy", "f2_giftig"],
    ascending=False,
).iloc[0]
accuracy_threshold = float(accuracy_threshold_row["threshold"])

threshold_choices = pd.DataFrame([
    selected_threshold_row.rename("veiligheid_min_85_recall"),
    accuracy_threshold_row.rename("hoogste_accuracy"),
])
print(f"Veiligheidsdrempel: {safety_threshold:.2f}")
print(f"Accuracy-drempel: {accuracy_threshold:.2f}")
display(threshold_choices.style.format("{:.3f}"))

plt.figure(figsize=(9, 5))
for metric in ["accuracy", "precision_giftig", "recall_giftig", "f2_giftig"]:
    plt.plot(threshold_results["threshold"], threshold_results[metric], label=metric)
plt.axvline(safety_threshold, color="firebrick", linestyle="--", label="veiligheidsdrempel")
plt.axvline(accuracy_threshold, color="black", linestyle="--", label="accuracy-drempel")
plt.axhline(TARGET_POISONOUS_RECALL, color="firebrick", linestyle=":", label="minimale recall")
plt.xlabel("Drempel voor klasse giftig")
plt.ylabel("Score")
plt.title("OOF-prestaties per beslisdrempel")
plt.legend()
plt.tight_layout()
plt.show()

### Beide vaste drempels controleren op validatie

Beide drempels worden zonder verdere aanpassing toegepast op de validatieset. Zo zien we afzonderlijk welke drempel de hoogste accuracy nastreeft en welke drempel minstens 85% recall voor giftig nastreeft. De definitieve keuze wordt nog niet gemaakt.

In [ ]:
accuracy_threshold_validation_metrics = evaluate_classifier(
    selected_model,
    X_validation,
    y_validation,
    "gradient_boosting_accuracy_drempel_validatie",
    threshold=accuracy_threshold,
)
safety_threshold_validation_metrics = evaluate_classifier(
    selected_model,
    X_validation,
    y_validation,
    "gradient_boosting_veiligheidsdrempel_validatie",
    threshold=safety_threshold,
)

threshold_comparison = pd.DataFrame([
    validation_metrics.rename("standaard_drempel"),
    accuracy_threshold_validation_metrics.rename("accuracy_drempel"),
    safety_threshold_validation_metrics.rename("veiligheidsdrempel"),
])
display(threshold_comparison.style.format("{:.3f}"))

## 8. Permutation importance

Histogram Gradient Boosting heeft geen eenvoudige ingebouwde feature importance. Daarom schudden we ieder oorspronkelijk kenmerk meerdere keren in de validatieset. De daling van F2 laat zien hoeveel het model op dat kenmerk steunt.

In [ ]:
f2_scorer = make_scorer(
    fbeta_score, beta=2, pos_label=POISONOUS_LABEL, zero_division=0
)
permutation = permutation_importance(
    selected_model,
    X_validation,
    y_validation,
    scoring=f2_scorer,
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
importance = (
    pd.DataFrame({
        "feature": X_validation.columns,
        "importance_mean": permutation.importances_mean,
        "importance_std": permutation.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
)
display(importance)
plt.figure(figsize=(8, 5))
plt.barh(
    importance["feature"][::-1],
    importance["importance_mean"][::-1],
    xerr=importance["importance_std"][::-1],
    color="darkorange",
)
plt.title("Permutation importance op validatie-F2")
plt.xlabel("Gemiddelde daling in F2")
plt.tight_layout()
plt.show()

## 9. Definitieve testset - voorlopig vergrendeld

Zet `RUN_FINAL_TEST` pas op `True` nadat één definitieve drempel is gekozen. Vul die drempel dan in bij `FINAL_THRESHOLD`; zolang deze `None` is kan de test niet per ongeluk worden uitgevoerd.

In [ ]:
FINAL_THRESHOLD = None

if RUN_FINAL_TEST and FINAL_THRESHOLD is None:
    raise ValueError("Kies eerst expliciet één definitieve drempel.")
elif RUN_FINAL_TEST:
    print(f"Definitief gekozen model: {selected_model_name}")
    test_metrics = evaluate_classifier(
        selected_model,
        X_test,
        y_test,
        "gradient_boosting_test",
        threshold=FINAL_THRESHOLD,
    )
else:
    print("Testset niet geëvalueerd: leg eerst alle modelkeuzes vast.")

## 10. Volgende beslissing

De hyperparameters en waarschijnlijkheidsdrempel zijn nu systematisch gekozen zonder de testset te gebruiken. Mogelijke vervolgstappen zijn de ruiskolommen gecontroleerd testen, een ensemble met Random Forest bouwen of daarna het model bevriezen en de testset één keer openen. Deze keuzes worden na overleg toegevoegd.